# SQL Analysis

## What this notebook does

Pulls three tables out of the database, each answering a different kind of question.

Everything downstream — the delivery analysis, the retention work, the dashboard — reads from these three files. Nothing else queries the database directly.

## The three tables

**1. Orders** (`order_level.csv`)
One row per order. Has the order's value, how many items were in it, and how late or early it arrived. This is what we use to ask whether slow delivery upsets customers.

**2. Customers** (`customer_level.csv`)
One row per customer. Has how recently they ordered, how often, how much they spent, and whether they ever came back for a second purchase. This is what we use for segmentation and retention.

**3. Top sellers** (`top_sellers.csv`)
The three highest-earning sellers in each state.

## One filter applied everywhere

Every query looks only at orders that were actually delivered.
Of the 99,441 orders in the dataset, 96,470 were delivered. The rest were cancelled, went unavailable, or never arrived. Those have no delivery date and no real customer experience attached, so including them would distort anything we measure about delivery or satisfaction.

In [7]:
import pandas as pd
import numpy as np
import sqlite3
import os

if os.path.exists('../olist.db'):
    db_path = '../olist.db' 
elif os.path.exists('olist.db'):
    db_path = 'olist.db'   
else:
    raise FileNotFoundError("Cannot find olist.db! Check your folder structure.")

print(f"Successfully located database at: {db_path}")

Successfully located database at: ../olist.db


## 1. Orders table

One row per delivered order.

**Why the query groups by order_id.** 
An order can contain several items, and the `order_items` table stores one row per item. Joining it directly would make a three-item order show up three times, which would inflate every count and average we calculate later. Grouping by order ID folds those rows back into one, adding up the item prices and counting how many items there were.

**Why reviews are not in this table.** 
A few hundred orders have more than one review attached to them. Joining reviews here would duplicate those orders all over again, undoing the grouping we just did. Reviews get attached later, in the analysis notebook, where we can handle the duplicates properly.

**Reading the delay column.** `delivery_delay_days` is the gap between when an order actually arrived and when it was promised. Positive means late, negative means early.
An order counts as **late** only if it arrived on a later calendar day than promised. The promised date is stored at midnight, so a simple "delay above zero" rule would mark an order delivered at 3pm on the promised day as late. That rule gave 8.1%; the corrected rate is 6.8%.

**Delay buckets.** Orders are also grouped by how late they arrived: early, on the promised day, 1–3 days late, 4–7, 8–14, and 15+. This lets the delivery analysis check whether satisfaction drops further the later an order is, rather than treating every late order the same. The boundaries were checked in notebook 03 so that every late bucket holds at least 1,300 orders.

In [8]:
conn = sqlite3.connect('../olist.db')

order_query = """
SELECT
    o.order_id,
    c.customer_unique_id,
    c.customer_state,
    o.order_purchase_timestamp,
    SUM(oi.price) AS order_value,
    SUM(oi.freight_value) AS freight_value,
    COUNT(*) AS n_items,
    JULIANDAY(o.order_delivered_customer_date)
        - JULIANDAY(o.order_estimated_delivery_date) AS delivery_delay_days,
    JULIANDAY(o.order_delivered_customer_date)
        - JULIANDAY(o.order_purchase_timestamp) AS delivery_time_days
FROM orders o
JOIN customers c   ON o.customer_id = c.customer_id
JOIN order_items oi ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
  AND o.order_delivered_customer_date IS NOT NULL
GROUP BY o.order_id;
"""

orders_df = pd.read_sql_query(order_query, conn)
conn.close()

# Late = arrived on a later calendar day than promised.
# Promised dates are stored at midnight, so "delay > 0" would wrongly flag orders delivered during the promised day itself.

orders_df['is_late'] = (orders_df['delivery_delay_days'] >= 1).astype(int)

# Group orders by how late they arrived (boundaries checked in notebook 03).
# right=False: each bucket includes its lower edge, so a delay of exactly 1.0 counts as "1-3 days late", matching the is_late rule above.

bucket_edges = [-np.inf, 0, 1, 4, 8, 15, np.inf]
bucket_names = ['Early', 'On promised day', '1-3 days late',
                '4-7 days late', '8-14 days late', '15+ days late']

orders_df['delay_bucket'] = pd.cut(orders_df['delivery_delay_days'],
                                   bins=bucket_edges,
                                   labels=bucket_names,
                                   right=False)

orders_df.to_csv('../data/order_level.csv', index=False)

print(f"Orders: {len(orders_df):,}")
print(f"Unique order_ids: {orders_df['order_id'].nunique():,}")
print(f"Late rate: {orders_df['is_late'].mean():.1%}")
orders_df.head()

Orders: 96,470
Unique order_ids: 96,470
Late rate: 6.8%


,order_id,customer_unique_id,customer_state,order_purchase_timestamp,order_value,freight_value,n_items,delivery_delay_days,delivery_time_days,is_late,delay_bucket
0,00010242fe8c5a6d1ba2dd792cb16214,871766c5855e863f6eccc05f988b23cb,RJ,2017-09-13 08:59:02,58.90,13.29,1,-8.011250,7.614421,0,Early
1,00018f77f2f0320c557190d7a144bdd3,eb28e67c4c0b83846050ddfb8a35d051,SP,2017-04-26 10:53:06,239.90,19.93,1,-2.330278,16.216181,0,Early
2,000229ec398224ef6ca0657da4fc703e,3818d81c6709e39d06b2738a8d3a2474,MG,2018-01-14 14:33:31,199.00,17.87,1,-13.444954,7.948437,0,Early
3,00024acbcdf0a6daa1e931b038114c75,af861d436cfc08b2c2ddefd0ba074622,SP,2018-08-08 10:00:35,12.99,12.79,1,-5.435660,6.147269,0,Early
4,00042b26cf59d7ce69dfabb4e55b4fd9,64b576fb70d441e8f1b2d7d446e483c5,SP,2017-02-04 13:57:51,199.90,18.14,1,-15.303808,25.114352,0,Early


## 2. Customers table

One row per customer, with everything we need for segmentation and retention.

### The repeat-purchase number is wrong, and this query fixes it

The obvious way to count repeat customers is to look for anyone with more than one order. Do that and you get 3.00%, which is the figure most public analyses of this dataset report.

That figure is wrong.

Olist is a marketplace. When someone buys from two different sellers in one checkout, the platform records it as two separate orders — same person, same moment, one shopping trip. Counting orders without checking the timestamps treats that as someone coming back.

Looking at the gap between a customer's first and second order shows a large cluster arriving within a single minute. Nobody decides to buy again in sixty seconds.

Requiring more than a day between orders reclassifies **808 customers — 29% of the apparent repeat buyers** — and brings the real repeat rate down to **2.13%**.

Both versions are kept in the output (`is_repeat_naive` and `is_repeat_customer`) so the difference is visible in the data rather than just claimed.

**Where the one-day cutoff came from.** 
Not a round number picked for convenience. The share of second orders barely moves between one minute and 24 hours, then starts climbing steadily after that. The cutoff sits in the flat stretch that separates split baskets from genuine returns.

### Why there's no frequency quartile

RFM segmentation normally scores customers on recency, frequency, and spend. Frequency doesn't work here: about 97% of customers ordered exactly once, so sorting them into four frequency buckets just splits identical values at random. The buckets would look like segments without meaning anything.

Recency is used instead, ordered so that quartile 4 is the most recent buyer.

In [9]:
conn = sqlite3.connect(db_path)

customer_query = """
WITH order_values AS (
    SELECT order_id, SUM(price) AS order_value
    FROM order_items
    GROUP BY order_id
),
customer_orders AS (
    SELECT
        c.customer_unique_id,
        o.order_id,
        o.order_purchase_timestamp,
        ov.order_value,
        MIN(o.order_purchase_timestamp) OVER (
            PARTITION BY c.customer_unique_id
        ) AS first_purchase_ts
    FROM orders o
    JOIN customers c    ON o.customer_id = c.customer_id
    JOIN order_values ov ON o.order_id = ov.order_id
    WHERE o.order_status = 'delivered'
),
customer_base AS (
    SELECT
        customer_unique_id,
        MIN(order_purchase_timestamp) AS first_purchase_date,
        MAX(order_purchase_timestamp) AS last_purchase_date,
        COUNT(DISTINCT order_id)      AS frequency,
        SUM(order_value)              AS monetary,
        -- Repeat only if a later order came MORE than a day after the first.
        -- Same-day extra orders are split multi-seller baskets, not return visits.
        MAX(CASE
                WHEN JULIANDAY(order_purchase_timestamp)
                     - JULIANDAY(first_purchase_ts) > 1
                THEN 1 ELSE 0
            END) AS is_repeat_customer,
        -- Naive definition, kept for comparison in the write-up
        CASE WHEN COUNT(DISTINCT order_id) > 1 THEN 1 ELSE 0 END AS is_repeat_naive
    FROM customer_orders
    GROUP BY customer_unique_id
),
with_recency AS (
    SELECT
        *,
        CAST(
            JULIANDAY((SELECT MAX(order_purchase_timestamp) FROM orders)) + 1
            - JULIANDAY(last_purchase_date)
        AS INTEGER) AS recency_days
    FROM customer_base
)
SELECT
    customer_unique_id,
    first_purchase_date,
    last_purchase_date,
    recency_days,
    frequency,
    monetary,
    is_repeat_customer,
    is_repeat_naive,
    NTILE(4) OVER (ORDER BY monetary ASC)      AS monetary_quartile,
    NTILE(4) OVER (ORDER BY recency_days DESC) AS recency_quartile
FROM with_recency;
"""

customers_df = pd.read_sql_query(customer_query, conn)

customers_df.to_csv('../data/customer_level.csv', index=False)

print(f"Customers: {len(customers_df):,}")
print(f"Naive repeat rate:     {customers_df['is_repeat_naive'].mean():.2%}")
print(f"Corrected repeat rate: {customers_df['is_repeat_customer'].mean():.2%}")
print(f"Customers reclassified: {(customers_df['is_repeat_naive'] - customers_df['is_repeat_customer']).sum():,}")
customers_df.head()

conn.close()


Customers: 93,358
Naive repeat rate:     3.00%
Corrected repeat rate: 2.13%
Customers reclassified: 808


## 3. Top sellers by state

The three highest-earning sellers in each state.

The useful part here is `PARTITION BY`, which restarts the ranking for every state. Without it we'd get the top three sellers in the whole country; with it we get the top three in each one.

Returns 56 rows across 22 states rather than 66 — several states have fewer than three sellers in the dataset.

In [10]:
conn = sqlite3.connect(db_path)

top_sellers_query = """
WITH seller_revenue AS (
    SELECT
        s.seller_state,
        s.seller_id,
        SUM(oi.price) AS total_revenue
    FROM sellers s
    JOIN order_items oi ON s.seller_id = oi.seller_id
    JOIN orders o       ON oi.order_id = o.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY s.seller_state, s.seller_id
),
ranked_sellers AS (
    SELECT
        seller_state,
        seller_id,
        total_revenue,
        RANK() OVER (PARTITION BY seller_state ORDER BY total_revenue DESC) AS state_rank
    FROM seller_revenue
)
SELECT *
FROM ranked_sellers
WHERE state_rank <= 3
ORDER BY seller_state ASC, state_rank ASC;
"""

top_sellers_df = pd.read_sql_query(top_sellers_query, conn)
conn.close()

top_sellers_df.to_csv('../data/top_sellers.csv', index=False)

print(f"Rows: {len(top_sellers_df)}  |  States: {top_sellers_df['seller_state'].nunique()}")
top_sellers_df.head(9)

Rows: 56  |  States: 22


,seller_state,seller_id,total_revenue,state_rank
0,AM,327b89b872c14d1c0be7235ef4871685,1177.00,1
1,BA,53243585a1d6dc2643021fd1853d8905,217940.44,1
2,BA,c72de06d72748d1a0dfb2125be43ba63,17522.00,2
3,BA,75d34ebb1bd0bd7dde40dd507b8169c3,12656.33,3
4,CE,bbf9ad41dca6603e614efcdad7aab8c4,7846.00,1
5,CE,dbdd0ec73a4817971d962698f2fea022,6384.00,2
6,CE,8d79c8a04e42d722a75097ce5cbcf2ef,2627.82,3
7,DF,44073f8b7e41514de3b7815dd0237f4f,18380.64,1
8,DF,f3b80352b986ab4d1057a4b724be19d0,9815.10,2


In [11]:
import os

os.makedirs('../sql', exist_ok=True)

queries = {
    'order_level.sql': order_query,
    'customer_level.sql': customer_query,
    'top_sellers.sql': top_sellers_query,
}

for filename, sql in queries.items():
    with open(f'../sql/{filename}', 'w') as f:
        f.write(sql.strip())
    print(f"Saved sql/{filename}")

Saved sql/order_level.sql
Saved sql/customer_level.sql
Saved sql/top_sellers.sql


In [12]:
import os
print(os.listdir('../sql'))

['.gitkeep', 'customer_level.sql', 'order_level.sql', 'top_sellers.sql']
